# Quantum Walk Fmo in native FlagQuantum

Complete local FMO Notebook: seven-site physical matrix, declared 35-term Pauli product formula, 20 quantum time points, 100 exact matrix-exponential time points, named populations, timings and comparison plot.

Reuse existing Observable and bounded Suzuki-Trotter capture. Preserve all coefficients, term order, time/repetitions, LSB target indexing and absolute phase. Classical parameters specialize at synthesis. Source bytes and licenses stay test-side; no cloud execution claim. Native real Observable replaces complex projector summation after exact cancellation; independent trace projections reconstruct the same padded physical matrix. torch.linalg.matrix_exp replaces scipy.linalg.expm for the exact classical reference; saved Matplotlib output replaces interactive display. Each of 20 finite-real specializations is synthesized locally; timing explicitly includes synthesis, execution and readout. Padding-state leakage is reported as approximation error, distinct from clean workspace.

Define the native quantum functions below, synthesize them locally, inspect the executed results, and compare the complete complex state with an independent Circuit.

See [the native Python entry](workflow.py).

## Native imports for main

In [ ]:
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import Output, QNum, allocate, qfunc, suzuki_trotter
from examples.synthesis.applications.chemistry.quantum_walk_fmo.physics import HAMILTONIAN

## main

In [ ]:
@qfunc
def main(qn: Output[QNum], time_point: float) -> None:
    allocate(3, qn)
    suzuki_trotter(HAMILTONIAN, time_point, 1, 10, qn)

## build program

In [ ]:
def build_program(time_point: float = 0.037) -> synth.Program:
    return synth.synthesize(main, classical_inputs={"time_point": time_point})

## Native imports for workflow

In [ ]:
import argparse
from collections.abc import Sequence
from pathlib import Path
from time import perf_counter
import torch
import flagquantum as fq
from examples.synthesis.applications.chemistry.quantum_walk_fmo.physics import SITE_HAMILTONIAN

## quantum sweep

In [ ]:
def quantum_sweep(times: Sequence[float]) -> tuple[torch.Tensor, tuple[float, ...]]:
    """Execute named site readout; timings include synthesis and execution."""
    populations = []
    durations = []
    for time_point in times:
        start = perf_counter()
        program = build_program(time_point)
        probabilities = dict(program.probabilities(fq.run(program), registers=("qn",)))
        populations.append([probabilities.get((site,), 0.0) for site in range(8)])
        durations.append(perf_counter() - start)
    return torch.tensor(populations, dtype=torch.float64), tuple(durations)

## exact sweep

In [ ]:
def exact_sweep(times: Sequence[float], initial_node: int = 0) -> torch.Tensor:
    """Use the seven-site dense matrix exponential, independently of synthesis."""
    if type(initial_node) is not int or not 0 <= initial_node < 7:
        raise ValueError("initial_node must be an integer in 0..6")
    hamiltonian = torch.tensor(SITE_HAMILTONIAN, dtype=torch.complex128)
    initial = torch.zeros(7, dtype=torch.complex128)
    initial[initial_node] = 1
    return torch.stack(
        [
            (torch.linalg.matrix_exp(-1j * time * hamiltonian) @ initial).abs().square()
            for time in times
        ]
    )

## save plot

In [ ]:
def save_plot(
    quantum_times: Sequence[float],
    quantum: torch.Tensor,
    exact_times: Sequence[float],
    exact: torch.Tensor,
    output: Path,
) -> None:
    """Plot the same first three sites as the source Notebook."""
    import matplotlib.pyplot as plt

    figure, axis = plt.subplots(figsize=(9, 5))
    for site, color, marker in zip(
        range(3), ("b", "g", "r"), ("s", "^", "x"), strict=True
    ):
        axis.plot(
            quantum_times,
            quantum[:, site].numpy(),
            color=color,
            marker=marker,
            linestyle="--",
            label=f"Site {site} (FlagQuantum product formula)",
        )
        axis.plot(
            exact_times,
            exact[:, site].numpy(),
            color=color,
            label=f"Site {site} (exact matrix exponential)",
        )
    axis.set(
        title="Continuous-time quantum walk with FMO Hamiltonian",
        xlabel="Time",
        ylabel="Population",
    )
    axis.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
    axis.grid(True, linestyle="--", alpha=0.7)
    figure.tight_layout()
    figure.savefig(output, dpi=150, bbox_inches="tight")
    plt.close(figure)

## run workflow

In [ ]:
def run_workflow(output: Path) -> None:
    """Run 20 quantum time points and 100 exact-reference time points."""
    quantum_times = tuple(index * 0.1 / 19 for index in range(20))
    exact_times = tuple(index * 0.1 / 99 for index in range(100))
    quantum, durations = quantum_sweep(quantum_times)
    exact = exact_sweep(exact_times)
    save_plot(quantum_times, quantum, exact_times, exact, output)
    reference = torch.nn.functional.pad(exact_sweep(quantum_times), (0, 1))
    errors = torch.linalg.vector_norm(quantum - reference, dim=1)
    print(f"Saved population comparison to {output}")
    print(
        f"Maximum probability L2 error against exact evolution: {float(errors.max()):.8g}"
    )
    print(f"Maximum padding-state population: {float(quantum[:, 7].max()):.8g}")
    print(f"Synthesis/execution/readout seconds at each quantum time: {durations}")

## Execute the complete native workflow

Run every cell in order. Any sampling is local and seeded; resource reports come from the actual native program. Read the reported approximation limits and optimization outcomes alongside the numerical results.

In [ ]:
output = Path("fmo-populations.png")

In [ ]:
run_workflow(output)

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(output)))

## Verify absolute amplitudes and clean workspace

The independent Circuit uses the same declared register order. The comparison checks every complex amplitude without global-phase alignment and verifies zero probability in hidden work qubits. This is a local numerical check.


In [ ]:
import torch
from flagquantum import synth
from examples.synthesis.compare_states import compare_states
from examples.synthesis.applications.chemistry.quantum_walk_fmo.comparisons.circuit import (
    reference_circuit,
)

for time_point in (0.037, 0.081):
    compare_states(build_program(time_point), reference_circuit(time_point), ("qn",))
print("Independent full-state comparison: passed")
